**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/senzai-2017"], check=True)
    os.chdir("/content/encore/entries/senzai-2017")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings, checked on DANDI:000003 version 0.261122.2143

This notebook belongs to the reconciliation of attempt `senzai-2017-02` (made blind, on version
0.261122.2143) with the earlier attempt `senzai-2017-01` (made on version 0.260218.2052), as
docs/protocol.md, "Repeat attempts", describes. Each section opens with one finding of the earlier
attempt, in its own words, and checks it on the files of this version. Where the blind attempt
already reports the same problem, the section says which of its findings, and shows only what
decides between still present and partly fixed.

Paper: Senzai Y, Buzsaki G (2017), Neuron 93, 691-704, doi:10.1016/j.neuron.2016.12.011.
Dataset: DANDI:000003, version 0.261122.2143 (doi:10.48324/dandi.000003/0.261122.2143), streamed;
nothing is downloaded. Loading uses the entry's helper `senzai.py`, unchanged.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import numpy as np
import pandas as pd
import senzai as S

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
print("DANDI:%s version %s" % (S.DANDISET, S.VERSION))


def text(x):
    x = x[()]
    return x.decode() if isinstance(x, bytes) else str(x)


def read_meta(asset):
    """Metadata of one file, every unit's label and spike-time order, and its burst index."""
    f = S.open_asset(asset["asset_id"])
    d = dict(path=asset["path"], file=asset["path"].split("ses-")[1].split("_")[0])
    d["subject"] = text(f["general/subject/subject_id"])
    d["genotype"] = text(f["general/subject/genotype"])
    d["subject_description"] = text(f["general/subject/description"])
    d["session_description"] = text(f["session_description"])
    u = f["units"]
    d["units_columns"] = tuple(u.attrs["colnames"])
    d["units_description"] = u.attrs["description"]
    d["cell_type_description"] = u["cell_type"].attrs["description"]
    d["waveforms_unit"] = u["waveforms"].attrs.get("unit") if "waveforms" in u else None
    d["electrodes_columns"] = tuple(f["general/extracellular_ephys/electrodes"].attrs["colnames"])
    d["cell_type"] = u["cell_type"][:].astype(str)
    d["region"] = u["region"][:].astype(str)
    d["shank"] = u["shank"][:]
    sti = u["spike_times_index"][:].astype(np.int64)
    st = u["spike_times"][:]
    s0 = np.concatenate([[0], sti[:-1]])
    d["n_spikes"] = sti - s0
    d["n_decreasing"] = np.array([int(np.sum(np.diff(st[a:b]) < 0)) for a, b in zip(s0, sti)])
    d["burst_index"] = np.array([S.burst_index(np.sort(st[a:b])) if S.TYPES.get(c, "other") != "other" else np.nan
                                 for a, b, c in zip(s0, sti, d["cell_type"])])
    del st
    d["span_s"] = f["processing/ecephys/LFP/lfp/data"].shape[0] / S.FS_LFP
    bs = f["processing/behavior/brain_states"]
    d["states_description"] = bs.attrs["description"]
    d["state_labels"] = tuple(sorted(set(bs["label"][:].astype(str))))
    d["tmaze"] = "processing/behavior/t_maze_position" in f
    if d["tmaze"]:
        g = f["processing/behavior/t_maze_position"]
        d["tmaze_series"] = {k: g[k].attrs.get("description", "") for k in g.keys()}
    d["light_pulses"] = None
    if "intervals/light_pulses" in f:
        lp = f["intervals/light_pulses"]
        d["light_pulses"] = (lp["start_time"][:], lp["protocol"][:].astype(str))
        d["light_pulses_description"] = lp.attrs["description"]
    d["monosynaptic_table"] = "processing/ecephys/monosynaptic_connections" in f
    d["wideband"] = "acquisition/ElectricalSeries" in f
    f.close()
    return d


assets = S.list_assets()
with ThreadPoolExecutor(12) as ex:
    M = list(ex.map(read_meta, assets))
M = sorted(M, key=lambda d: d["path"])
units = pd.concat([pd.DataFrame(dict(file=d["file"], subject=d["subject"], cell_type=d["cell_type"], region=d["region"],
                                     shank=d["shank"], n_spikes=d["n_spikes"], n_decreasing=d["n_decreasing"],
                                     burst_index=d["burst_index"])) for d in M], ignore_index=True)
units["type"] = units.cell_type.map(S.TYPES).fillna("other")


In [ ]:
print(len(M), "files,", len({d["subject"] for d in M}), "subjects,", len(units), "units")
print("files with the wide-band signal:", sum(d["wideband"] for d in M), "; with T-maze positions:", sum(d["tmaze"] for d in M))
print("spikes per unit: median", int(units.n_spikes.median()), "; units with a burst index:", int(units.burst_index.notna().sum()))
print(units.groupby("subject").size().to_string())

## Finding 1

> In 24 files the end index of the first unit in the units table is short by a multiple of 65,536
> spikes (k = 1 to 7). The missing spikes open the second unit's spike train, which then restarts at
> the beginning of the recording, so the second unit's times are not sorted. The first moved spike
> follows the first unit's last stored spike by 2 ms to 0.5 s. This looks like an integer overflow at
> conversion; the waveforms follow the same boundaries.

The second unit's train restarting would show as a decrease in its spike times. Every unit's times
are checked for any decrease.

In [ ]:
dec = units[units.n_decreasing > 0]
print("units whose spike times ever decrease:", len(dec), "of", len(units), "; files:", dec.file.nunique())
first_two = units.groupby("file").head(2)
print("first two units of each file, spike times decreasing:", int((first_two.n_decreasing > 0).sum()), "of", len(first_two))

## Finding 2

> The genotypes in the files (6 SST-Cre, 5 Drd2-Cre::Ai35, 4 POMC-Cre::Arch, 1 POMC-Cre::Ai35) do
> not match the paper's 16 mice (7 SST-Cre, 5 DRD2-Cre::Ai35, 3 POMC-Cre::Ai35 and 1 wild type).

In [ ]:
subj = pd.DataFrame([dict(subject=d["subject"], genotype=d["genotype"], description=d["subject_description"]) for d in M]).drop_duplicates("subject")
print(subj.groupby("genotype").subject.apply(list).to_string())
print()
for s in ["YutaMouse41", "YutaMouse57"]:
    print(s, ":", subj.set_index("subject").description[s])
print()
print("The dandiset's description, as published:")
import json, urllib.request
with urllib.request.urlopen(f"{S.API}/dandisets/{S.DANDISET}/versions/{S.VERSION}/") as r:
    print(json.load(r)["description"])

## Finding 3

> 83 of 101 files do not say which shank each unit was recorded on (`shank_id` is a cluster number,
> and there is no `electrode_group`), and no file gives site coordinates (x, y, z are NaN). The
> paper's depth measures can be computed for 17 files only, and distances between cells not at all.

The blind attempt reports that no position is given for the sites (its finding 1). Shown here is
what has changed: the shank of each unit.

In [ ]:
print("units-table layouts (columns):")
for cols, n in Counter(d["units_columns"] for d in M).items():
    print(" ", n, "files:", ", ".join(cols))
print("electrodes-table columns:", Counter(d["electrodes_columns"] for d in M))
print("units without a shank (shank < 1):", int((units.shank < 1).sum()))

## Finding 4

> The T-maze position series (`twhl_linearized`, `twhl_norm`) are not described. The values show
> column 0 of `twhl_linearized` runs along the central arm (0 to 2) and column 1 is an angle along the
> circular side arms, negative for the left arm, which fits the maze in the Methods but is not
> stated.

In [ ]:
tm = [d for d in M if d["tmaze"]]
print(len(tm), "files with T-maze positions; series and descriptions in the first:")
for k, v in tm[0]["tmaze_series"].items():
    print(f"- {k}: {v}")
print("files whose series all carry a description:", sum(all(v for v in d["tmaze_series"].values()) for d in tm))

## Finding 5

> The states table holds `awake`, `quiet`, `nrem`, `rem` and `transit` without a description; the
> paper names wake, NREM and REM, and whether `quiet` counts as wake is not stated.

In [ ]:
print(M[0]["states_description"])
print()
print(Counter(d["state_labels"] for d in M))

## Finding 6

> The interneuron labels (narrow-waveform, wide-waveform) do not say whether the unit is in the
> dentate gyrus or CA3, which the paper separates; only the excitatory label was split by region
> (granule cell or CA3 pyramidal cell).

In [ ]:
paper = units[units.subject != S.EXCLUDED_SUBJECT]
print(pd.crosstab(paper.cell_type, paper.region).to_string())
print("\n(the paper: 248 narrow-waveform and 268 wide-waveform dentate interneurons)")

## Finding 7

> The monosynaptic connections the authors detected and curated (`MonoSynConvClick.mat`) were not
> converted (the conversion code comments them out), so the pair-based results (Figures 3, 4 and 7)
> cannot be checked on the authors' pairs.

The blind attempt reports the files that lack the table (its finding 4).

In [ ]:
has = [d["file"] for d in M if d["monosynaptic_table"]]
print("files with the authors' monosynaptic_connections table:", len(has), "of", len(M))
print("without:", ", ".join(d["file"] for d in M if not d["monosynaptic_table"]))

## Finding 8

> The dandiset holds 568 mossy cells and 609 granule cells against the paper's 500 and 586, and 237
> units labelled `missing` (units of the spike sorting with no row in the authors' unit summary).

In [ ]:
print(pd.crosstab(units.cell_type, units.subject == S.EXCLUDED_SUBJECT).rename(columns={False: "paper's 15 mice", True: S.EXCLUDED_SUBJECT}).to_string())
print()
print("cell_type description:", M[0]["cell_type_description"])

## Finding 9

> The light-pulse series PulseStim_5V_20ms_LD12 of YutaMouse57-161006 holds 89,086 times of which
> 24,286 are duplicates, and its times decrease once.

In [ ]:
rows = []
for d in M:
    if d["light_pulses"] is None:
        continue
    t, prot = d["light_pulses"]
    for p in np.unique(prot):
        x = t[prot == p]
        rows.append(dict(file=d["file"], protocol=p, n=len(x), duplicates=len(x) - len(np.unique(x)), decreases=int(np.sum(np.diff(x) < 0))))
lp = pd.DataFrame(rows)
print("light-pulse series:", len(lp), "in", lp.file.nunique(), "files; with duplicates:", int((lp.duplicates > 0).sum()),
      "; with decreasing times:", int((lp.decreases > 0).sum()))
print(lp[lp.file == "YutaMouse57-161006"].to_string(index=False))
print()
print([d for d in M if d["file"] == "YutaMouse57-161006"][0]["light_pulses_description"])

## Finding 10

> The units table has two layouts. 18 files (mice 33, 41, 42 and 54) carry an `electrode_group`
> column naming each unit's shank, one of them without waveforms (YutaMouse33-150218, which the
> conversion script lists as excluded); the other 83 files do not.

In [ ]:
for cols, n in Counter(d["units_columns"] for d in M).items():
    files = [d["file"] for d in M if d["units_columns"] == cols]
    print(n, "files:", ", ".join(cols), "" if n > 5 else files)

## Finding 11

> Seven dates hold two files of one mouse (the second named with `b`). They share no spike trains, so
> they are different recordings, but the metadata do not say whether they come from a second probe
> or a second session.

The blind attempt reports that the files do not say whether the two files of a day hold the same
neurons (its finding 11). Shown here is what the files now say of the 'b' files.

In [ ]:
for d in M:
    if "b16" in d["file"]:
        print(d["file"], ":", d["session_description"])

## Finding 12

> The Methods give a burst index threshold of 1.2 for excitatory cells, and the Figure 1A legend
> gives 1.8.

The two numbers are the paper's, and the dataset cannot change them. The labels in the files may
show which threshold the authors' classification used: the share of excitatory and wide-waveform
units whose burst index (as the Methods define it, `senzai.burst_index`) lies between the two.

In [ ]:
x = units[(units.subject != S.EXCLUDED_SUBJECT) & units.type.isin(["GC", "MC", "CA3", "wI"]) & units.burst_index.notna()].copy()
x["group"] = np.where(x.type == "wI", "wide-waveform interneuron", "GC, MC or CA3")
x["band"] = pd.cut(x.burst_index, [-np.inf, 1.2, 1.8, np.inf], labels=["<= 1.2", "1.2 to 1.8", "> 1.8"])
print(pd.crosstab(x.group, x.band, margins=True).to_string())

## Findings 13 and 14

> 13. In the 17 files with the wide-band signal and each unit's shank, the electrodes table's
> assignment of columns to shanks and sites disagrees with the data. The column where a unit's spike
> is largest in the wide-band signal lies on the shank the units table gives for 45% of units (11% to
> 79% by file) and is the table's own site for 9%, and the DS2 amplitude along the table's site
> numbers crosses zero more than once on 43 of 64 shanks where it crosses at all. The wide-band and
> LFP columns match each other. The units' shanks (`electrode_group`) and stored waveform channels fit
> the wide-band signal under one fixed map of columns, the same in all 4 mice (a map from three mice
> gives the column of a unit's largest spike for 78% to 89% of units in the fourth, against 5% to 13%
> for the table), so the error is in the table, whose blocks of 8 rows do not match the probe's
> columns.

> 14. The stored spike waveforms are about 5 times larger than the same spikes in the wide-band
> signal (median peak-to-peak 421 against 81 uV, 300 Hz high-pass), and their filtering and scaling
> are not described. They match the wide-band spike best after a 300 Hz high-pass (median correlation
> 0.987, against 0.981 unfiltered and 0.867 at 800 Hz), with a gain of 5.08, close to 5.13, the
> inverse of the wide-band conversion of 0.195 uV per count, which is consistent with waveforms
> stored in ADC counts and read as microvolts. Each holds 1.6 ms with the trough at 0.8 ms, which cuts
> off the peak of 43% of GC, 67% of MC and 78% of CA3 waveforms.

For GC, MC and CA3 units in the files with the wide-band signal, the column where the mean 300 Hz
high-passed spike is largest (`senzai.wideband_sites`, one 52 s block per file, to keep the reads
short) is compared with the column of the largest stored waveform through the electrodes table, and
with the unit's shank. For finding 14, the peak-to-peak amplitude of the stored mean waveform, in
the units the files give, is compared with that of the high-passed wide-band mean at the same column,
in the wide band's units. The blind attempt reports the truncated window and the undocumented
filtering (its finding 3); what is checked here is the scale.

In [ ]:
def wb_check(asset):
    f = S.open_asset(asset["asset_id"])
    if "acquisition/ElectricalSeries" not in f:
        f.close()
        return []
    u = f["units"]
    ct = u["cell_type"][:].astype(str)
    shank = u["shank"][:]
    sti = u["spike_times_index"][:].astype(np.int64)
    st = u["spike_times"][:]
    s0 = np.concatenate([[0], sti[:-1]])
    keep = [k for k in range(len(ct)) if S.TYPES.get(ct[k]) in ("GC", "MC", "CA3")]
    spikes = {k: np.sort(st[s0[k]:sti[k]]) for k in keep}
    del st
    elec = u["electrodes"][:]
    ei = u["electrodes_index"][:].astype(np.int64)
    e0 = np.concatenate([[0], ei[:-1]])
    wb_el = f["acquisition/ElectricalSeries/electrodes"][:]
    col_of = {int(e): j for j, e in enumerate(wb_el)}
    gname = f["general/extracellular_ephys/electrodes/group_name"][:].astype(str)
    wb_uv = float(f["acquisition/ElectricalSeries/data"].attrs["conversion"]) * 1e6
    wf_unit = u["waveforms"].attrs.get("unit", "") if "waveforms" in u else ""
    sites = S.wideband_sites(f, spikes, n_blocks=1)
    out = []
    for k in keep:
        col, n, w, _ = sites[k]
        if col < 0:
            continue
        r = dict(file=asset["path"].split("ses-")[1].split("_")[0], type=S.TYPES[ct[k]], wb_col=col,
                 same_shank=gname[int(wb_el[col])] == f"shank{int(shank[k])}", wb_p2p_uv=float(np.ptp(w)) * wb_uv,
                 wb_uv_per_step=wb_uv, waveforms_unit=wf_unit)
        mw = S.mean_waveform(u, k) if "waveforms_index_index" in u else None
        if mw is not None:
            ch = int(np.argmax(mw.max(axis=1) - mw.min(axis=1)))
            unit_el = elec[e0[k]:ei[k]]
            r["table_col"] = col_of.get(int(unit_el[ch]), -1) if ch < len(unit_el) else -1
            r["stored_p2p_steps"] = float(np.ptp(mw[ch]))
        out.append(r)
    f.close()
    return out


wb_assets = [a for a, d in zip(sorted(assets, key=lambda a: a["path"]), M) if d["wideband"]]
with ThreadPoolExecutor(6) as ex:
    W = pd.DataFrame([r for rs in ex.map(wb_check, wb_assets) for r in rs])
print(len(wb_assets), "files with the wide-band signal;", len(W), "GC, MC and CA3 units with a wide-band site")
print("wide band, uV per step:", W.wb_uv_per_step.unique(), "; stored waveforms' unit:", W.waveforms_unit.unique())

In [ ]:
# Finding 13: the electrodes table's map of columns against the wide-band signal
t = W[W.table_col.notna()]
print("largest wide-band spike on the unit's shank (units.shank):", f"{W.same_shank.mean():.3f}", f"({int(W.same_shank.sum())} of {len(W)})")
print("largest wide-band spike at the table's site of the largest stored waveform:", f"{(t.wb_col == t.table_col).mean():.3f}",
      f"({int((t.wb_col == t.table_col).sum())} of {len(t)})")
print("by file, fraction at the table's site:")
print(t.groupby("file").apply(lambda g: round((g.wb_col == g.table_col).mean(), 2), include_groups=False).to_string())

In [ ]:
# Finding 14: the scale of the stored waveforms against the wide band
t = W[W.stored_p2p_steps.notna()].copy()
t["stored_p2p_uv"] = t.stored_p2p_steps * S.UV_PER_STEP
t["ratio"] = t.stored_p2p_uv / t.wb_p2p_uv
print("median peak-to-peak: stored %.0f uV (%.0f steps), wide band 300 Hz high-passed %.0f uV" %
      (t.stored_p2p_uv.median(), t.stored_p2p_steps.median(), t.wb_p2p_uv.median()))
print("ratio stored / wide band: median %.2f (10th to 90th percentile %.2f to %.2f)" %
      (t.ratio.median(), t.ratio.quantile(0.1), t.ratio.quantile(0.9)))

## Summary

Each section's printout is the evidence for the earlier finding's status in the `reconciliation`
block of `attempts/senzai-2017-02.yaml`.